# Phase 4 — LoRA fine-tune the retriever's embedding model

Fine-tunes `all-MiniLM-L6-v2` (the bi-encoder in `src/embed.py`) with LoRA on
query→chunk triplets, then **merges** the adapter and exports a drop-in model.

**Runtime → Change runtime type → T4 GPU** before running.

| Step | File |
|------|------|
| Upload to Colab | `train/triplets.jsonl` (from your repo) |
| Download from Colab | `minilm-lora.zip` → unzip to `models/minilm-lora/` |

The training papers are **disjoint from the eval papers** — that's what makes
the before/after number meaningful rather than memorisation.

## 1. Install

In [ ]:
!pip -q install -U sentence-transformers peft datasets accelerate

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > T4 GPU')

## 2. Upload `triplets.jsonl`

Run this cell, then pick `train/triplets.jsonl` from your repo.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose train/triplets.jsonl

import json
rows = [json.loads(l) for l in open('triplets.jsonl', encoding='utf-8')]
print(f'{len(rows)} triplets from papers:', sorted({r["paper"] for r in rows}))
print('\nexample query   :', rows[0]['query'])
print('example positive:', rows[0]['positive'][:120], '...')
print('example negative:', rows[0]['negative'][:120], '...')

## 3. Hold out a slice to sanity-check training

The real metric is `run_eval` back in the repo. This split only tells us the
loss is going down on data it didn't train on — cheap insurance against a
silently broken run.

In [ ]:
import random
from datasets import Dataset

random.Random(0).shuffle(rows)
split = max(1, int(0.1 * len(rows)))
eval_rows, train_rows = rows[:split], rows[split:]

def to_ds(rs):
    # MultipleNegativesRankingLoss reads the columns in order: anchor, positive, negative
    return Dataset.from_list([
        {'anchor': r['query'], 'positive': r['positive'], 'negative': r['negative']}
        for r in rs
    ])

train_ds, eval_ds = to_ds(train_rows), to_ds(eval_rows)
print('train', len(train_ds), '| eval', len(eval_ds))

## 4. Attach LoRA

Only the adapter trains — the base model stays frozen. `query/key/value` are the
attention projections in MiniLM's BERT backbone.

In [ ]:
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType

BASE = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(BASE)

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['query', 'key', 'value'],
)

# ST's own PEFT integration: injects LoRA in place and freezes the base weights.
# Don't try get_peft_model + merge_and_unload here -- assigning a PeftModel onto
# model[0].auto_model does not stick in sentence-transformers v5, and the
# backbone stays a plain BertModel. ST can load an adapter directly (step 6),
# so no merging is needed.
model.add_adapter(peft_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)  <- should be ~1-2%')

## 5. Train

`MultipleNegativesRankingLoss` pulls each query toward its positive and pushes it
away from its hard negative *and* every other positive in the batch — so a bigger
batch means more negatives and a stronger signal. A few hundred triplets on a T4
takes ~1–3 minutes.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

args = SentenceTransformerTrainingArguments(
    output_dir='out',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,          # higher than full fine-tuning: only the adapter moves
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='no',
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=MultipleNegativesRankingLoss(model),
)
trainer.train()

## 6. Save + export

Saves the **adapter** (a few MB), not a full model copy. `sentence-transformers`
detects `adapter_config.json` on load and applies the LoRA over the base model —
so no merging is needed. The repo has `peft` installed for this.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer as ST

# Save. With the adapter active this writes adapter_config.json + adapter
# weights (a few MB) rather than a full copy of the base model -- ST resolves
# the base from adapter_config.json's base_model_name_or_path on load.
model.save('minilm-lora')
!ls -la minilm-lora

# Smoke test: reload from disk exactly as the repo will.
reloaded = ST('minilm-lora')
q = 'what optimizer does the paper propose?'
v = reloaded.encode([q], normalize_embeddings=True)
print('\nembedding shape:', v.shape, '| non-zero:', bool(v.any()))
assert v.shape[-1] == 384 and v.any(), 'reloaded model produced a bad embedding'

# Did training actually change anything? cosine of 1.0 => the LoRA is a no-op
base_v = ST(BASE).encode([q], normalize_embeddings=True)
print('cosine vs base :', round(float(np.dot(v[0], base_v[0])), 4), ' (1.0 would mean UNCHANGED)')

!zip -qr minilm-lora.zip minilm-lora
!du -sh minilm-lora.zip

## 7. Download

Unzip in the repo so that `models\minilm-lora\adapter_config.json` exists.

**Rebuild the index for each model.** A query embedded by one model can't be
compared against vectors built by another, so flipping `LORA` at eval time alone
is meaningless — rebuild, then eval, per side:

```bat
:: BEFORE — base model
set LORA=0 & python -m scripts.rebuild_all & python -m eval.run_eval

:: AFTER — LoRA
set LORA=1 & python -m scripts.rebuild_all & python -m eval.run_eval
```

**Known gotcha:** ST saves the adapter with `base_model.model.` prefixed keys and
no adapter name, but loads expecting `...lora_A.default.weight`. Mismatched keys
are silently re-initialised (`lora_B = 0`), so the LoRA becomes a **no-op that
looks like it loaded**. Always check the `cosine vs base` in step 6 — if it's
`1.0`, the adapter did nothing. `scripts/fix_adapter_keys.py` renames the keys.

In [ ]:
from google.colab import files
files.download('minilm-lora.zip')